In [22]:
import pandas as pd
import numpy as np

# Load the training data
train_df = pd.read_csv('dataset/train.csv')

print("Dataset Shape:", train_df.shape)
print("\nFirst few rows:")
print(train_df.head())

Dataset Shape: (59611, 24)

First few rows:
   founder_id  founder_age founder_gender  years_with_startup founder_role  \
0        8410           31           Male                  19    Education   
1       64756           59         Female                   4        Media   
2       30257           24         Female                  10   Healthcare   
3       65791           36         Female                   7    Education   
4       65026           56           Male                  41    Education   

   monthly_revenue_generated work_life_balance_rating venture_satisfaction  \
0                     5390.0                Excellent               Medium   
1                     5534.0                     Poor                 High   
2                     8159.0                     Good                 High   
3                     3989.0                     Good                 High   
4                     4821.0                      NaN                  NaN   

  startup_performa

In [13]:
# Check basic info and data types
print("Data Types:")
print(train_df.dtypes)
print("\n" + "="*50)
print("\nBasic Statistics:")
print(train_df.describe())

Data Types:
founder_id                      int64
founder_age                     int64
founder_gender                 object
years_with_startup              int64
founder_role                   object
monthly_revenue_generated     float64
work_life_balance_rating       object
venture_satisfaction           object
startup_performance_rating     object
funding_rounds_led              int64
working_overtime               object
distance_from_investor_hub      int64
education_background           object
personal_status                object
num_dependents                float64
startup_stage                  object
team_size_category             object
years_since_founding          float64
remote_operations              object
leadership_scope               object
innovation_support             object
startup_reputation             object
founder_visibility             object
retention_status               object
dtype: object


Basic Statistics:
         founder_id   founder_age  years_w

In [25]:
# Check 1: Age-related inconsistencies
print("="*70)
print("CHECK 1: AGE-RELATED INCONSISTENCIES")
print("="*70)

# Years with startup cannot exceed founder age
invalid_years = train_df[train_df['years_with_startup'] > train_df['founder_age']]
print(f"\n1.1 Founders with years_with_startup > founder_age: {len(invalid_years)}")
if len(invalid_years) > 0:
    print(invalid_years[['founder_id', 'founder_age', 'years_with_startup']].head(10))

# Company founded before founder was born (truly impossible)
impossible_founding = train_df[train_df['years_since_founding'] > train_df['founder_age']]
print(f"\n1.2 Company founded before founder was born: {len(impossible_founding)}")
if len(impossible_founding) > 0:
    print(impossible_founding[['founder_id', 'founder_age', 'years_since_founding']].head(10))

# Calculate founding age and check for unusually young founders
train_df['calculated_founding_age'] = train_df['founder_age'] - train_df['years_since_founding']
very_young_founders = train_df[train_df['calculated_founding_age'] < 10]
print(f"\n1.3 Founders who started company before age 10: {len(very_young_founders)}")
if len(very_young_founders) > 0:
    print(very_young_founders[['founder_id', 'founder_age', 'years_since_founding', 'calculated_founding_age']].head(10))

# Statistical analysis of founding age
founding_age_stats = train_df['calculated_founding_age'].describe()
print(f"\n1.4 Founding age statistics:")
print(founding_age_stats)

# Unusual age values (too young or too old for startup founders)
unusual_age = train_df[(train_df['founder_age'] < 18) | (train_df['founder_age'] > 80)]
print(f"\n1.5 Founders with unusual current age (< 18 or > 80): {len(unusual_age)}")
if len(unusual_age) > 0:
    print(unusual_age[['founder_id', 'founder_age', 'years_with_startup', 'years_since_founding']].head(10))

CHECK 1: AGE-RELATED INCONSISTENCIES

1.1 Founders with years_with_startup > founder_age: 3
       founder_id  founder_age  years_with_startup
50279       50946           13                  22
51116        4762            6                   9
56103       34398           23                  26

1.2 Company founded before founder was born: 39355
    founder_id  founder_age  years_since_founding
0         8410           31                  89.0
2        30257           24                  74.0
3        65791           36                  50.0
4        65026           56                  68.0
5        24368           38                  47.0
6        64970           47                  93.0
7        36999           48                  88.0
8        32714           57                  75.0
9        15944           24                  45.0
11        9063           29                  38.0

1.3 Founders who started company before age 10: 46380
    founder_id  founder_age  years_since_foundi

## Decision Analysis: Drop Column vs Remove Rows

Should we drop `years_since_founding` column entirely, or remove problematic rows?

In [27]:
print("="*70)
print("OPTION 1: DROP 'years_since_founding' COLUMN")
print("="*70)

# Calculate correlation with target variable
from scipy.stats import chi2_contingency, f_oneway

# Check if years_since_founding has predictive power
correlation_with_target = train_df[['years_since_founding', 'retention_status']].dropna()
stayed = correlation_with_target[correlation_with_target['retention_status'] == 'Stayed']['years_since_founding']
left = correlation_with_target[correlation_with_target['retention_status'] == 'Left']['years_since_founding']

# Perform t-test
from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(stayed, left, nan_policy='omit')

print(f"\nStatistical relationship with target variable:")
print(f"Mean years_since_founding for 'Stayed': {stayed.mean():.2f}")
print(f"Mean years_since_founding for 'Left': {left.mean():.2f}")
print(f"T-test p-value: {p_value:.6f}")
print(f"Statistically significant? {'YES' if p_value < 0.05 else 'NO'}")

print(f"\nIf we DROP the column:")
print(f"✓ Keep all {len(train_df)} rows")
print(f"✓ Lose 1 feature (years_since_founding)")
print(f"✗ Lose potential predictive information" if p_value < 0.05 else "✓ Column may not be very predictive anyway")

print("\n" + "="*70)
print("OPTION 2: REMOVE ROWS WITH IMPOSSIBLE FOUNDING AGES")
print("="*70)

# Option 2a: Remove only truly impossible cases (founded before born)
impossible_mask = train_df['years_since_founding'] > train_df['founder_age']
valid_impossible = train_df[~impossible_mask]

print(f"\n2a. Remove only 'founded before born' cases:")
print(f"   Rows to remove: {impossible_mask.sum()} ({impossible_mask.sum()/len(train_df)*100:.2f}%)")
print(f"   Remaining rows: {len(valid_impossible)} ({len(valid_impossible)/len(train_df)*100:.2f}%)")

# Option 2b: Remove founded before age 10
very_young_mask = train_df['calculated_founding_age'] < 10
valid_young = train_df[~very_young_mask]

print(f"\n2b. Remove 'founded before age 10' cases:")
print(f"   Rows to remove: {very_young_mask.sum()} ({very_young_mask.sum()/len(train_df)*100:.2f}%)")
print(f"   Remaining rows: {len(valid_young)} ({len(valid_young)/len(train_df)*100:.2f}%)")

# Option 2c: Keep only reasonable founding ages (18-65)
reasonable_mask = (train_df['calculated_founding_age'] >= 18) & (train_df['calculated_founding_age'] <= 65)
valid_reasonable = train_df[reasonable_mask]

print(f"\n2c. Keep only reasonable founding ages (18-65):")
print(f"   Rows to remove: {(~reasonable_mask).sum()} ({(~reasonable_mask).sum()/len(train_df)*100:.2f}%)")
print(f"   Remaining rows: {len(valid_reasonable)} ({len(valid_reasonable)/len(train_df)*100:.2f}%)")

print("\n" + "="*70)
print("RECOMMENDATION")
print("="*70)

OPTION 1: DROP 'years_since_founding' COLUMN

Statistical relationship with target variable:
Mean years_since_founding for 'Stayed': 56.52
Mean years_since_founding for 'Left': 54.95
T-test p-value: 0.000000
Statistically significant? YES

If we DROP the column:
✓ Keep all 59611 rows
✓ Lose 1 feature (years_since_founding)
✗ Lose potential predictive information

OPTION 2: REMOVE ROWS WITH IMPOSSIBLE FOUNDING AGES

2a. Remove only 'founded before born' cases:
   Rows to remove: 39355 (66.02%)
   Remaining rows: 20256 (33.98%)

2b. Remove 'founded before age 10' cases:
   Rows to remove: 46380 (77.80%)
   Remaining rows: 13231 (22.20%)

2c. Keep only reasonable founding ages (18-65):
   Rows to remove: 54783 (91.90%)
   Remaining rows: 4828 (8.10%)

RECOMMENDATION


In [28]:
print("\nBased on the data quality issues:")
print(f"\n1. The column has {'SIGNIFICANT' if p_value < 0.05 else 'NO SIGNIFICANT'} relationship with target")
print(f"2. {impossible_mask.sum()/len(train_df)*100:.1f}% of data is truly impossible (founded before born)")
print(f"3. {very_young_mask.sum()/len(train_df)*100:.1f}% has suspicious values (founded before age 10)")
print(f"4. Only {len(valid_reasonable)/len(train_df)*100:.1f}% has reasonable founding ages (18-65)")

print("\n" + "-"*70)
if p_value < 0.05:
    print("RECOMMENDED APPROACH: Remove rows with impossible/suspicious values")
    print("\nReason: The column has predictive power, so we should keep it.")
    print("Remove only the most problematic rows (founded before age 10).")
    print(f"This keeps {len(valid_young)} rows ({len(valid_young)/len(train_df)*100:.1f}%) with valid data.")
else:
    print("RECOMMENDED APPROACH: DROP the 'years_since_founding' column")
    print("\nReason: Column doesn't significantly predict retention and has")
    print(f"poor quality data ({very_young_mask.sum()/len(train_df)*100:.1f}% suspicious values).")
    print("Better to keep all rows and use other features for prediction.")

print("-"*70)


Based on the data quality issues:

1. The column has SIGNIFICANT relationship with target
2. 66.0% of data is truly impossible (founded before born)
3. 77.8% has suspicious values (founded before age 10)
4. Only 8.1% has reasonable founding ages (18-65)

----------------------------------------------------------------------
RECOMMENDED APPROACH: Remove rows with impossible/suspicious values

Reason: The column has predictive power, so we should keep it.
Remove only the most problematic rows (founded before age 10).
This keeps 13231 rows (22.2%) with valid data.
----------------------------------------------------------------------


In [15]:
# Check 2: Revenue inconsistencies
print("\n" + "="*70)
print("CHECK 2: REVENUE INCONSISTENCIES")
print("="*70)

# Negative revenue
negative_revenue = train_df[train_df['monthly_revenue_generated'] < 0]
print(f"\n2.1 Data points with negative revenue: {len(negative_revenue)}")
if len(negative_revenue) > 0:
    print(negative_revenue[['founder_id', 'monthly_revenue_generated', 'startup_stage']].head(10))

# Extremely high revenue (potential outliers - > 99.9th percentile)
revenue_99_9 = train_df['monthly_revenue_generated'].quantile(0.999)
extreme_revenue = train_df[train_df['monthly_revenue_generated'] > revenue_99_9]
print(f"\n2.2 Extremely high revenue (> 99.9th percentile ${revenue_99_9:.2f}): {len(extreme_revenue)}")
if len(extreme_revenue) > 0:
    print(extreme_revenue[['founder_id', 'monthly_revenue_generated', 'startup_stage', 'years_since_founding']].head(10))

# Zero or very low revenue for senior stage startups
low_revenue_senior = train_df[(train_df['startup_stage'] == 'Senior') & (train_df['monthly_revenue_generated'] < 1000)]
print(f"\n2.3 Senior stage startups with very low revenue (< $1000): {len(low_revenue_senior)}")
if len(low_revenue_senior) > 0:
    print(low_revenue_senior[['founder_id', 'monthly_revenue_generated', 'startup_stage', 'years_since_founding']].head(10))


CHECK 2: REVENUE INCONSISTENCIES

2.1 Data points with negative revenue: 0

2.2 Extremely high revenue (> 99.9th percentile $13716.33): 58
       founder_id  monthly_revenue_generated startup_stage  \
387         46092                    15495.0           Mid   
1776         6468                    13961.0         Entry   
3521        71311                    14014.0         Entry   
3609        49640                    14016.0         Entry   
7229          803                    45180.0         Entry   
8346        16017                    14176.0         Entry   
8575        41653                    13962.0           Mid   
8809        35278                    14276.0           Mid   
9798        63208                    36770.0         Entry   
10962        1306                    14066.0         Entry   

       years_since_founding  
387                    73.0  
1776                   92.0  
3521                   75.0  
3609                   32.0  
7229                    9.0

In [16]:
# Check 3: Startup stage and years inconsistencies
print("\n" + "="*70)
print("CHECK 3: STARTUP STAGE INCONSISTENCIES")
print("="*70)

# Entry stage with many years since founding
entry_old = train_df[(train_df['startup_stage'] == 'Entry') & (train_df['years_since_founding'] > 10)]
print(f"\n3.1 Entry stage startups with > 10 years since founding: {len(entry_old)}")
if len(entry_old) > 0:
    print(entry_old[['founder_id', 'startup_stage', 'years_since_founding', 'monthly_revenue_generated']].head(10))

# Senior stage with very few years
senior_young = train_df[(train_df['startup_stage'] == 'Senior') & (train_df['years_since_founding'] < 5)]
print(f"\n3.2 Senior stage startups with < 5 years since founding: {len(senior_young)}")
if len(senior_young) > 0:
    print(senior_young[['founder_id', 'startup_stage', 'years_since_founding', 'monthly_revenue_generated']].head(10))

# Years with startup > years since founding
invalid_timeline = train_df[train_df['years_with_startup'] > train_df['years_since_founding']]
print(f"\n3.3 Years_with_startup > years_since_founding: {len(invalid_timeline)}")
if len(invalid_timeline) > 0:
    print(invalid_timeline[['founder_id', 'years_with_startup', 'years_since_founding']].head(10))


CHECK 3: STARTUP STAGE INCONSISTENCIES

3.1 Entry stage startups with > 10 years since founding: 21722
    founder_id startup_stage  years_since_founding  monthly_revenue_generated
6        64970         Entry                  93.0                     3681.0
7        36999         Entry                  88.0                    11223.0
8        32714         Entry                  75.0                     3773.0
9        15944         Entry                  45.0                     7319.0
10       29972         Entry                  17.0                     5443.0
12       21896         Entry                  68.0                     9039.0
15       17696         Entry                  21.0                     5176.0
22       58984         Entry                  20.0                     8079.0
25       20748         Entry                  23.0                     4987.0
27       65251         Entry                  67.0                     7571.0

3.2 Senior stage startups with < 5 ye

In [17]:
# Check 4: Missing values in critical fields
print("\n" + "="*70)
print("CHECK 4: MISSING VALUES")
print("="*70)

missing_summary = train_df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
print(f"\nColumns with missing values:")
print(missing_summary)

# Check rows with multiple missing values
multiple_missing = train_df[train_df.isnull().sum(axis=1) > 2]
print(f"\n4.1 Rows with more than 2 missing values: {len(multiple_missing)}")
if len(multiple_missing) > 0:
    print(multiple_missing[['founder_id', 'work_life_balance_rating', 'venture_satisfaction', 
                            'num_dependents', 'team_size_category', 'years_since_founding']].head(10))


CHECK 4: MISSING VALUES

Columns with missing values:
work_life_balance_rating     10144
venture_satisfaction          7164
num_dependents                4780
years_since_founding          4184
team_size_category            2992
monthly_revenue_generated     1800
dtype: int64

4.1 Rows with more than 2 missing values: 4780
     founder_id work_life_balance_rating venture_satisfaction  num_dependents  \
6         64970                      NaN                  NaN             NaN   
34        15737                      NaN                  NaN             NaN   
35        51448                      NaN                  NaN             NaN   
54        66579                      NaN                  NaN             NaN   
70        68364                      NaN                  NaN             NaN   
80         7424                      NaN                  NaN             NaN   
86        60485                      NaN                  NaN             NaN   
95        23371           

In [18]:
# Check 5: Logical inconsistencies
print("\n" + "="*70)
print("CHECK 5: LOGICAL INCONSISTENCIES")
print("="*70)

# Negative number of dependents
negative_dependents = train_df[train_df['num_dependents'] < 0]
print(f"\n5.1 Negative number of dependents: {len(negative_dependents)}")
if len(negative_dependents) > 0:
    print(negative_dependents[['founder_id', 'num_dependents', 'personal_status']].head(10))

# Extremely high number of dependents (> 10 seems unusual)
high_dependents = train_df[train_df['num_dependents'] > 10]
print(f"\n5.2 Unusually high number of dependents (> 10): {len(high_dependents)}")
if len(high_dependents) > 0:
    print(high_dependents[['founder_id', 'num_dependents', 'founder_age', 'personal_status']].head(10))

# Distance from investor hub
negative_distance = train_df[train_df['distance_from_investor_hub'] < 0]
print(f"\n5.3 Negative distance from investor hub: {len(negative_distance)}")
if len(negative_distance) > 0:
    print(negative_distance[['founder_id', 'distance_from_investor_hub']].head(10))

# Negative funding rounds
negative_funding = train_df[train_df['funding_rounds_led'] < 0]
print(f"\n5.4 Negative funding rounds: {len(negative_funding)}")
if len(negative_funding) > 0:
    print(negative_funding[['founder_id', 'funding_rounds_led', 'startup_stage']].head(10))


CHECK 5: LOGICAL INCONSISTENCIES

5.1 Negative number of dependents: 0

5.2 Unusually high number of dependents (> 10): 0

5.3 Negative distance from investor hub: 0

5.4 Negative funding rounds: 0


In [19]:
# Check 6: Duplicate entries
print("\n" + "="*70)
print("CHECK 6: DUPLICATE ENTRIES")
print("="*70)

# Check for duplicate founder_ids
duplicate_ids = train_df[train_df.duplicated(subset=['founder_id'], keep=False)]
print(f"\n6.1 Duplicate founder_ids: {len(duplicate_ids)}")
if len(duplicate_ids) > 0:
    print(duplicate_ids[['founder_id', 'founder_age', 'founder_role', 'retention_status']].head(20))

# Check for completely duplicate rows
duplicate_rows = train_df[train_df.duplicated(keep=False)]
print(f"\n6.2 Completely duplicate rows: {len(duplicate_rows)}")
if len(duplicate_rows) > 0:
    print(duplicate_rows.head(10))


CHECK 6: DUPLICATE ENTRIES

6.1 Duplicate founder_ids: 26
       founder_id  founder_age founder_role retention_status
4448        71428           30   Healthcare             Left
6072        26393           37   Technology             Left
8305        17110           38      Finance           Stayed
9249         1973           28        Media             Left
10130        3940           23   Healthcare             Left
12358       54760           37   Technology           Stayed
15550       41745           18      Finance             Left
26253       18701           12    Education           Stayed
31132       23782           55      Finance             Left
37300        7984           47      Finance             Left
47848       17350           29    Education           Stayed
52577       47308           46    Education             Left
54954        6829           20    Education           Stayed
59598       17350           29    Education           Stayed
59599       41745         

In [20]:
# Check 7: Category value inconsistencies
print("\n" + "="*70)
print("CHECK 7: CATEGORICAL VALUE CHECKS")
print("="*70)

# Check unique values for each categorical column
categorical_columns = ['founder_gender', 'founder_role', 'work_life_balance_rating', 
                       'venture_satisfaction', 'startup_performance_rating', 'working_overtime',
                       'education_background', 'personal_status', 'startup_stage', 
                       'team_size_category', 'remote_operations', 'leadership_scope',
                       'innovation_support', 'startup_reputation', 'founder_visibility',
                       'retention_status']

for col in categorical_columns:
    unique_vals = train_df[col].unique()
    print(f"\n{col}: {len(unique_vals)} unique values")
    print(f"  Values: {sorted([str(v) for v in unique_vals if pd.notna(v)])}")


CHECK 7: CATEGORICAL VALUE CHECKS

founder_gender: 2 unique values
  Values: ['Female', 'Male']

founder_role: 5 unique values
  Values: ['Education', 'Finance', 'Healthcare', 'Media', 'Technology']

work_life_balance_rating: 5 unique values
  Values: ['Excellent', 'Fair', 'Good', 'Poor']

venture_satisfaction: 5 unique values
  Values: ['High', 'Low', 'Medium', 'Very High']

startup_performance_rating: 4 unique values
  Values: ['Average', 'Below Average', 'High', 'Low']

working_overtime: 2 unique values
  Values: ['No', 'Yes']

education_background: 5 unique values
  Values: ['Associate Degree', 'Bachelor’s Degree', 'High School', 'Master’s Degree', 'PhD']

personal_status: 3 unique values
  Values: ['Divorced', 'Married', 'Single']

startup_stage: 3 unique values
  Values: ['Entry', 'Mid', 'Senior']

team_size_category: 4 unique values
  Values: ['Large', 'Medium', 'Small']

remote_operations: 2 unique values
  Values: ['No', 'Yes']

leadership_scope: 2 unique values
  Values: ['N

In [26]:
# Summary of all issues found
print("\n" + "="*70)
print("SUMMARY OF DATA QUALITY ISSUES")
print("="*70)

# Create a set to track unique problematic rows
problematic_indices = set()

# Count all issues and track indices
invalid_years_indices = train_df[train_df['years_with_startup'] > train_df['founder_age']].index
invalid_years_count = len(invalid_years_indices)
problematic_indices.update(invalid_years_indices)

impossible_founding_indices = train_df[train_df['years_since_founding'] > train_df['founder_age']].index
impossible_founding_count = len(impossible_founding_indices)
problematic_indices.update(impossible_founding_indices)

very_young_founders_indices = train_df[train_df['calculated_founding_age'] < 10].index
very_young_founders_count = len(very_young_founders_indices)
problematic_indices.update(very_young_founders_indices)

unusual_age_indices = train_df[(train_df['founder_age'] < 18) | (train_df['founder_age'] > 80)].index
unusual_age_count = len(unusual_age_indices)
problematic_indices.update(unusual_age_indices)

negative_revenue_indices = train_df[train_df['monthly_revenue_generated'] < 0].index
negative_revenue_count = len(negative_revenue_indices)
problematic_indices.update(negative_revenue_indices)

entry_old_indices = train_df[(train_df['startup_stage'] == 'Entry') & (train_df['years_since_founding'] > 10)].index
entry_old_count = len(entry_old_indices)
problematic_indices.update(entry_old_indices)

senior_young_indices = train_df[(train_df['startup_stage'] == 'Senior') & (train_df['years_since_founding'] < 5)].index
senior_young_count = len(senior_young_indices)
problematic_indices.update(senior_young_indices)

invalid_timeline_indices = train_df[train_df['years_with_startup'] > train_df['years_since_founding']].index
invalid_timeline_count = len(invalid_timeline_indices)
problematic_indices.update(invalid_timeline_indices)

negative_dependents_indices = train_df[train_df['num_dependents'] < 0].index
negative_dependents_count = len(negative_dependents_indices)
problematic_indices.update(negative_dependents_indices)

high_dependents_indices = train_df[train_df['num_dependents'] > 10].index
high_dependents_count = len(high_dependents_indices)
problematic_indices.update(high_dependents_indices)

negative_distance_indices = train_df[train_df['distance_from_investor_hub'] < 0].index
negative_distance_count = len(negative_distance_indices)
problematic_indices.update(negative_distance_indices)

negative_funding_indices = train_df[train_df['funding_rounds_led'] < 0].index
negative_funding_count = len(negative_funding_indices)
problematic_indices.update(negative_funding_indices)

duplicate_ids_indices = train_df[train_df.duplicated(subset=['founder_id'], keep=False)].index
duplicate_ids_count = len(duplicate_ids_indices)
problematic_indices.update(duplicate_ids_indices)

missing_total = train_df.isnull().sum().sum()

print(f"\n1. Age Inconsistencies:")
print(f"   - Years with startup > age: {invalid_years_count}")
print(f"   - Company founded before founder born: {impossible_founding_count}")
print(f"   - Founded company before age 10: {very_young_founders_count}")
print(f"   - Unusual current age (< 18 or > 80): {unusual_age_count}")

print(f"\n2. Revenue Issues:")
print(f"   - Negative revenue: {negative_revenue_count}")

print(f"\n3. Timeline Issues:")
print(f"   - Entry stage with > 10 years: {entry_old_count}")
print(f"   - Senior stage with < 5 years: {senior_young_count}")
print(f"   - Years with startup > years since founding: {invalid_timeline_count}")

print(f"\n4. Logical Issues:")
print(f"   - Negative dependents: {negative_dependents_count}")
print(f"   - > 10 dependents: {high_dependents_count}")
print(f"   - Negative distance: {negative_distance_count}")
print(f"   - Negative funding rounds: {negative_funding_count}")

print(f"\n5. Data Quality:")
print(f"   - Duplicate founder IDs: {duplicate_ids_count}")
print(f"   - Total missing values: {missing_total}")

print(f"\n" + "="*70)
print(f"UNIQUE DATA POINTS WITH AT LEAST ONE ISSUE: {len(problematic_indices)}")
print(f"PERCENTAGE OF DATASET: {len(problematic_indices)/len(train_df)*100:.2f}%")
print(f"TOTAL DATASET SIZE: {len(train_df)} rows")
print("="*70)
print(f"\nNote: The sum of individual issues ({invalid_years_count + impossible_founding_count + very_young_founders_count + unusual_age_count + negative_revenue_count + entry_old_count + senior_young_count + invalid_timeline_count + negative_dependents_count + high_dependents_count + negative_distance_count + negative_funding_count + duplicate_ids_count}) is higher because many data points have multiple issues.")


SUMMARY OF DATA QUALITY ISSUES

1. Age Inconsistencies:
   - Years with startup > age: 3
   - Company founded before founder born: 39355
   - Founded company before age 10: 46380
   - Unusual current age (< 18 or > 80): 6

2. Revenue Issues:
   - Negative revenue: 0

3. Timeline Issues:
   - Entry stage with > 10 years: 21722
   - Senior stage with < 5 years: 27
   - Years with startup > years since founding: 0

4. Logical Issues:
   - Negative dependents: 0
   - > 10 dependents: 0
   - Negative distance: 0
   - Negative funding rounds: 0

5. Data Quality:
   - Duplicate founder IDs: 26
   - Total missing values: 35248

UNIQUE DATA POINTS WITH AT LEAST ONE ISSUE: 49553
PERCENTAGE OF DATASET: 83.13%
TOTAL DATASET SIZE: 59611 rows

Note: The sum of individual issues (107519) is higher because many data points have multiple issues.
